<a href="https://colab.research.google.com/github/shemo203/calmrocks-personal/blob/main/01-model-apis/00-prompting-basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompting Fundamentals

**Goal:** The four techniques that get more out of a model *before* you reach for any API feature: clear instructions, examples, output-format specs, and step-by-step reasoning. Each one is shown moving a real number, not a vibe.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## What prompting is *for*

Before the techniques, be clear about *why* you invest in the prompt at all. A good prompt isn't about sounding clever to the model. It's the cheapest lever on three things you'll otherwise pay for in code, latency, or dollars:

| Goal | What a better prompt buys | Where the course goes deep |
|---|---|---|
| **Quality & reliability** | the right answer *consistently*, not just once: fewer wrong outputs, less drift across calls | measured properly in **evals** (sections 02, 04) |
| **Formatting / structure** | output your *program* can consume, a fixed shape rather than prose to regex | prompting gets you close; **01-structured-output** makes it a guarantee |
| **Cost & latency** | the same result in **fewer tokens**. A tight prompt that states the task plainly beats a rambling one that over-explains, and skips output you'll throw away | budgeted in **04-context-and-caching** |

That last one is underrated: **a precise prompt is often a cheaper prompt.** "Reply with one word" instead of letting the model write a paragraph, or asking only for the fields you'll use, cuts output tokens on *every* call, which multiplies across millions of them. Prompt quality and cost control are the same lever pulled from two sides.

The rest of this notebook is the *how*: four techniques that serve those goals, each shown moving a real number.

## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client, the only dependency this notebook needs.
2. **Load your API key.** Free key at [console.groq.com](https://console.groq.com/) (no credit card); in Colab add it via the **key icon** → **Add new secret** named exactly `GROQ_API_KEY`, notebook access on. Locally, set `GROQ_API_KEY` in your environment.

(Full walkthrough: [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [21]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [22]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready Groq client.
client, MODEL = setup()

Groq client ready. MODEL = openai/gpt-oss-120b


## The prompt is your primary API surface

Before RAG, before agents, before fine-tuning, the cheapest and fastest lever on a model's behavior is the words you send it. That's *prompting*, and it's worth being deliberate about because you'll do it inside every other technique in this course: RAG stuffs retrieved text into a prompt, an agent's whole personality is a prompt, a judge is a prompt.

Two things to get out of the way first, because "prompt engineering" attracts nonsense:

> **🚩 Common mistake —** cargo-cult prompting: magic phrases ("you are a world-class expert", "I'll tip you $200", "take a deep breath"). Some helped on older models and mostly wash out on current ones. Don't collect incantations. Learn the few techniques that have a *mechanism*, and keep the ones that move your number.

And the discipline that makes all of this real, carried from **section 02 (evals)**: a prompt change is a *measured* change. "It looked better on the one example I tried" is how you ship a prompt that's worse on the other ninety. Every technique below is shown with a before/after you could turn into an eval. This notebook eyeballs the difference to teach the mechanism; in practice you'd score it against a golden set.

## Technique 1: say exactly what you want

The single biggest lever, and the most underused. Models are literal-ish: vague instructions get vague, inconsistent output. Specify the **task, the format, and the constraints** explicitly. Watch a lazy prompt vs. a specified one on the same input.

In [23]:
review = ("Bought the standing desk. Assembly took two people and the instructions "
          "were useless, but it's rock solid and the motor is whisper quiet. "
          "Shipping was a week late though.")

def ask(prompt):
    r = client.chat.completions.create(
        model=MODEL, max_tokens=200,
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

print("--- vague:")
print(ask(f"Extract, from the product review below, the sentiment in one word per aspect\n\n{review}"))

print("\n--- specified (task + format + constraint):")
print(ask(
    "Extract, from the product review below, the aspects mentioned and the sentiment "
    "for each. Output one line per aspect as `aspect: sentiment` where sentiment is "
    "positive, negative, or mixed. Only aspects the review actually mentions.\n\n"
    f"{review}"))

--- vague:
-

--- specified (task + format + constraint):
assembly: negative  
instructions: negative  
stability: positive  
motor


The vague prompt gives a paragraph you'd have to parse by hand; the specified one gives structured, consistent lines. **Most "the model can't do this" problems are really "I didn't tell it precisely what I wanted."** Fixing the instruction is free and should always be your first move, before few-shot, before tools, before anything.

The three things worth stating explicitly almost every time:
- **Task:** what to do, in a verb ("classify", "extract", "rewrite").
- **Format:** the exact shape of the output (one per line, JSON, a single word).
- **Constraints:** the boundaries ("only what's mentioned", "≤ 20 words", "no preamble").

Notice the specified prompt is also the *cheaper* one: it asks for terse lines, not a paragraph, so it spends fewer output tokens per call. Constraints like "one line per aspect" or "reply with one word" are a quality control **and** a cost control at once, the same lever from two sides (the token math is section 04).

## Technique 2: few-shot, or show don't just tell

When a task is hard to *describe* but easy to *demonstrate* (a specific label set, an output style, an edge-case convention), give the model a few examples of input → output. This is **few-shot prompting** (zero-shot = no examples; few-shot = a handful). Examples pin down behavior that prose struggles to.

Below: a niche classification (routing support tickets to internal teams) where the label meanings aren't obvious from the names alone. Zero-shot guesses; few-shot locks it in.

In [29]:
SYSTEM = "Route each ticket to exactly one team. Reply with only the team name."
teams = "Teams: BILLING, PLATFORM (outages/API), DEVICE (hardware), TRUST (fraud/abuse)."

def route(ticket, shots=""):
    r = client.chat.completions.create(
        model=MODEL, max_tokens=500,
        messages=[{"role": "system", "content": f"{SYSTEM} {teams}"},
                  {"role": "user", "content": f"{shots}Ticket: {ticket}\nTeam:"}])
    return r.choices[0].message.content.strip()

# A genuinely ambiguous ticket: "charged after cancelling" could read as BILLING,
# but here the convention is that suspected-fraud charges go to TRUST.
ticket = "I cancelled last month but was charged again, and I don't recognize the device that logged in."

print("zero-shot:", route(ticket))

FEWSHOT = (
    "Ticket: My invoice is higher than my plan.\nTeam: BILLING\n"
    "Ticket: Charges I don't recognize plus a login from another country.\nTeam: TRUST\n"
    "Ticket: The API returns 503 for every request.\nTeam: PLATFORM\n"
)

FEWSHOT2 = (
    "Ticket: I need to change my phone number. \nTeam: BILLING\n",
    "Ticket: I got a password change request but i haven't ordered it? \nTeam: TRUST\n",
    "Ticket: The application is returning a 404 error on all API calls. \nTeam: PLATFORM\n",
    "Ticket: My laptop screen is flickering when I move it. \nTeam: DEVICE\n",
    "Ticket: The server is down, and I can't access any of my data. \nTeam: PLATFORM\n"
)
print("few-shot :", route(ticket, shots=FEWSHOT2))

zero-shot: TRUST
few-shot : TRUST


With one example of the fraud-routing convention, the model applies it to the new ticket. Notes on using few-shot well:

- **Examples teach patterns you can't easily state:** tone, edge-case handling, a label taxonomy. If you *can* state the rule cleanly, prefer a clear instruction (technique 1); it's cheaper.
- **Cover the boundaries, not just the easy cases.** The example that matters is the ambiguous one.
- **Few-shot costs tokens on every call** (the examples ride in each prompt). That's the tradeoff vs. fine-tuning the behavior in, the decision you'll make in section 06.
- Structured output (next notebook) often *replaces* format examples: the schema pins the shape, so your few-shot can focus on judgment.

## Technique 3: specify the output format (the bridge to everything structured)

If a downstream program consumes the output, you must constrain its shape. Prose is for humans; your code wants a fixed format. Asking for it plainly gets you most of the way, and reveals *why* the next notebook exists.

In [25]:
bug = ("App crashes on launch after the 4.2 update, but only on Android 13. "
       "Started yesterday. My coworker on iOS is fine.")

print(ask(
    "Summarize this bug report as JSON with keys: title (short string), "
    "platform (string), severity (one of: low, medium, high). "
    "Output only the JSON, no other text.\n\n" + bug))

{
  "title": "Crash on launch after 4.2 update on Android 13",
  "platform": "Android 13",
  "severity": "high"
}


That usually works, and *usually* is the problem. Run it a few times: sometimes you get a markdown ```json fence, a "Here's the JSON:" preamble, or a paraphrased key. Prompting for format is **necessary but not sufficient** when a program is parsing the result.

> **⭐ Key takeaway —** clear format instructions get you 90% of the way; the last 10% (guaranteed-parseable, schema-valid output) needs the API-level tools, JSON mode and tool/function calling, which is exactly what **[01-structured-output](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/01-model-apis/01-structured-output.ipynb)** builds next. Prompting is the floor; structured output is the guarantee.

## Technique 4: ask for reasoning before the answer

For anything requiring a few steps (arithmetic, logic, multi-constraint decisions), telling the model to **work through it before answering** ("chain of thought") measurably improves accuracy. It gives the model tokens to compute in rather than forcing a one-shot guess. The catch: the reasoning is now *in the output*, so you either keep it or extract the final answer.

A word problem the model can flub if it answers immediately, with vs. without step-by-step:

In [26]:
q = ("A team of 3 reviews 90 tickets. Two reviewers handle 25 each; the third takes "
     "the rest. If the third reviewer works twice as fast and clears 8 tickets an hour, "
     "how many hours does the third reviewer need? Give the final number.")

print("--- answer-first (no reasoning room):")
print(ask(q + " Reply with just the number."))

print("\n--- reasoning first, then answer:")
print(ask(q + " Think step by step, then end with 'ANSWER: <number>'."))

--- answer-first (no reasoning room):


--- reasoning first, then answer:



The step-by-step version is more reliable, and it puts the answer behind a parseable marker (`ANSWER:`) so your code can still extract one field. Two caveats that keep this from being a reflex:

- **It costs tokens and latency:** you're generating the reasoning. Don't add it to tasks that don't need it (simple classification, extraction).
- **Modern "reasoning" models do this internally.** On those, an explicit "think step by step" is often redundant or even counterproductive. As always: this is an eval question, not an article of faith. Measure whether it helps on *your* task and model.

## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| State task + format + constraints explicitly; fix the instruction first | "The model can't do this" when you never told it precisely what to do |
| Few-shot when behavior is easier shown than described; cover the edge cases | Pile on examples for a rule you could state in one clear sentence |
| Specify output format, and use structured output (nb 01) when a program parses it | Trust prose "output JSON" for a downstream parser |
| Add step-by-step reasoning where it earns its tokens; extract the final field | Bolt CoT onto every call, or onto a reasoning model that already does it |
| Ask only for the tokens you'll use (terse output, needed fields), cheaper per call | A rambling prompt + open-ended output you then truncate or ignore |
| Judge every prompt change against a golden set (section 02) | Tune prompts by eyeballing one example; cargo-cult incantations |

(The full stack-wide list lives in [docs/best-practices-and-anti-patterns.md](https://github.com/calmrocks/ai-engineer-notebooks/blob/main/docs/best-practices-and-anti-patterns.md).)

## Exercises

1. **Instruction tightening.** Take the vague review prompt from technique 1 and add constraints one at a time (format, max length, "mentioned aspects only"). Run each version 3× and note which constraint removed the most variation. Which single addition helped most?
2. **Zero- vs few-shot, measured.** Write 8 ambiguous tickets with known correct teams (a tiny golden set, section 02 style). Compute routing accuracy zero-shot vs with your 3 examples. Does few-shot actually win, and by how much? Then try 6 examples: does more help or plateau?
3. **When CoT backfires.** Find a *simple* task (single-label sentiment) and compare accuracy and token cost with vs. without "think step by step." Confirm the reasoning adds cost without accuracy, the evidence for *not* making it a reflex.
4. **Prompt as the cheapest lever.** Pick something the model gets wrong, and get it right using *only* prompt changes (instruction, examples, format, reasoning), no code, no tools. Log what worked. This is the habit: exhaust the free lever before reaching for the expensive ones (structured output, RAG, fine-tuning) in the notebooks ahead.